# Finalize Gene Clusters (manual merge variant)

**Optional / human-judgment path.** This notebook produces ONE `manual_merge` finalize variant. The buildable variants (`direct` / `auto_merge` / `grid`) need no notebook — Snakemake builds them. Use this only for a variant declared as `type: manual_merge` in `config/analysis.yaml` → `clustering.variants`, where you eyeball the `n_intermediate`→`final_n_clusters` merge.

## Inputs (from the prepare spine)
- `results/clustering/{dataset}/_work/annotated_data.parquet`  (annotated fitting table)
- `results/clustering/{dataset}/_work/scaled_data.parquet`  (scaled DR/DL matrix)

The notebook clusters the scaled matrix to `N_INTERMEDIATE` with `METHOD` itself (same call the buildable variants use — there is no shared candidate stage), so it works for any method.

## Output
- `resources/curated/final_clusters/{dataset}/{variant}.tsv`  (version-controlled; consumed by enrichment.smk + ml.smk when this variant is selected). Uses the unified `cluster` column for the final labels (same contract as the buildable variants) plus a `raw_cluster` column holding the `N_INTERMEDIATE` pre-merge labels.

**This is a human-judgment step.** You maintain ONE dict (`merge_groups`) mapping each raw transitional id (0..N_INTERMEDIATE-1) to a merge-group label. Final numbering is then computed automatically by the shared `renumber_by_dr` (lowest mean DR = WT) — you no longer hand-assign final ids. Re-run against the current dataset and adjust `merge_groups` by looking at the review plot before writing the output.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parents[1]))  # repo root
from workflow.src.clustering.candidates import cluster_one_method, renumber_by_dr
from workflow.src.io import read_parquet
from workflow.src.plotting.gene_level import visualize_cluster_on_feature_space

In [ ]:
# --- Config: which dataset / method / variant you are finalizing ---
DATASET = "HD_DIT_HAP"   # switch to your target dataset
METHOD = "kmeans"                       # clustering method for the transitional labels
N_INTERMEDIATE = 64                     # transitional cluster count (match the variant's config)
VARIANT = "kmeans_merge9_manual"        # must match a manual_merge variant in analysis.yaml
WT_CLUSTER = 9                          # final id for the lowest-DR (WT) group
FINAL_N_CLUSTERS = 9
RANDOM_STATE = 42

REPO = Path.cwd().parents[1]
WORK = REPO / f"results/clustering/{DATASET}/_work"
OUTPUT = REPO / f"resources/curated/final_clusters/{DATASET}/{VARIANT}.tsv"

# Read the annotated table + scaled matrix from the spine, then cluster the scaled
# matrix to N_INTERMEDIATE transitional clusters ourselves (same call the buildable
# variants use — there is no shared candidate stage).
annotated = read_parquet(WORK / "annotated_data.parquet")
scaled = read_parquet(WORK / "scaled_data.parquet")
raw_labels = pd.Series(
    cluster_one_method(METHOD, scaled, N_INTERMEDIATE, RANDOM_STATE), index=scaled.index
)
data_df = annotated.copy()
data_df["raw_cluster"] = raw_labels
data_df.shape, sorted(raw_labels.unique())[:10]

## 1. Review the transitional clusters

In [ ]:
# Only genes that were actually clustered have a raw_cluster (others are NaN).
clustered = data_df.dropna(subset=["raw_cluster"]).copy()
clustered["raw_cluster"] = clustered["raw_cluster"].astype(int)
fig = visualize_cluster_on_feature_space(clustered, 'raw_cluster')
plt.show()

## 2. Merge transitional clusters → groups (single dict), then auto-number by DR

Maintain the ONE dict below: raw transitional id (0..N_INTERMEDIATE-1) → an arbitrary merge-group label. Group labels can be any integers — they only define *which transitional clusters merge together*. Final ids are then assigned automatically by mean DR (`renumber_by_dr`), so you never hand-number.

The dict below is the HD kmeans starting point (N_INTERMEDIATE=64). **Adjust by eye** against the plot above for the current dataset/method — cluster geometry differs, so this mapping is unlikely to be correct elsewhere. Every raw id present in the data must appear as a key, and the number of distinct group labels must equal `FINAL_N_CLUSTERS`.

In [ ]:
# raw transitional id (0..N_INTERMEDIATE-1) -> arbitrary merge-group label (HD kmeans, N_INTERMEDIATE=64).
merge_groups = {
    0: 3, 1: 1, 2: 2, 3: 3, 4: 4, 5: 3, 6: 6, 7: 1, 8: 3, 9: 9,
    10: 3, 11: 11, 12: 1, 13: 13, 14: 11, 15: 4, 16: 4, 17: 1, 18: 3, 19: 1,
    20: 6, 21: 6, 22: 2, 23: 3, 24: 1, 25: 11, 26: 1, 27: 3, 28: 28, 29: 3,
    30: 28, 31: 11, 32: 1, 33: 11, 34: 13, 35: 3, 36: 4, 37: 9, 38: 4, 39: 3,
    40: 6, 41: 13, 42: 3, 43: 6, 44: 2, 45: 11, 46: 28, 47: 3, 48: 11, 49: 1,
    50: 1, 51: 28, 52: 9, 53: 3, 54: 1, 55: 1, 56: 3, 57: 3, 58: 6, 59: 3,
    60: 3, 61: 1, 62: 3, 63: 2,
}

# Map each clustered gene to its merge group, then auto-number by mean DR (WT = lowest).
group_labels = clustered["raw_cluster"].map(merge_groups)
assert group_labels.notna().all(), "every raw id in the data must be a key in merge_groups"
assert group_labels.nunique() == FINAL_N_CLUSTERS, (
    f"merge_groups has {group_labels.nunique()} distinct groups, expected {FINAL_N_CLUSTERS}"
)
final = renumber_by_dr(clustered, group_labels, n_clusters=FINAL_N_CLUSTERS, wt_cluster=WT_CLUSTER)

# Write final labels back onto the full annotated frame (unclustered genes stay NaN).
data_df["cluster"] = final.reindex(data_df.index)
data_df.loc[clustered.index, "cluster"].value_counts().sort_index()

## 3. Review the merged 1..9 clusters (WT = 9)

In [ ]:
plot_df = data_df.dropna(subset=["cluster"]).copy()
plot_df["cluster"] = plot_df["cluster"].astype(int)
fig = visualize_cluster_on_feature_space(
    plot_df, 'cluster', show_box=True, legend=True, cluster_minus_one=True
)
plt.show()

## 4. Write the curated hub artifact

Only run this once you are satisfied with the merge above. This file is version-controlled and consumed by enrichment + ml.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
data_df.to_csv(OUTPUT, sep="\t", index=True)
print(f"Wrote {len(data_df)} genes ({data_df['cluster'].notna().sum()} clustered) to {OUTPUT}")